### **SVM Baseline Model**

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Download required NLTK data (run once)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

In [ ]:
# Load and combine the two files
fake_df = pd.read_csv('../data/raw/Fake.csv')
true_df = pd.read_csv('../data/raw/True.csv')

fake_df['label'] = 1  # Fake = 1
true_df['label'] = 0  # Real = 0

fake_df['full_text'] = fake_df['title'] + " " + fake_df['text']
true_df['full_text'] = true_df['title'] + " " + true_df['text']

df = pd.concat([fake_df[['full_text', 'label']], true_df[['full_text', 'label']]], ignore_index=True)
print(f"Total samples: {len(df)}")
print(f"Class distribution:\n{df['label'].value_counts()}")

In [ ]:
# Basic text cleaning function
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # Remove numbers, punctuation
    text = re.sub(r'\s+', ' ', text)      # Collapse whitespace
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text.strip()

print("Cleaning text...")
df['clean_text'] = df['full_text'].apply(clean_text)

In [ ]:
# Train-test split (70/15/15 as per roadmap)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['clean_text'], df['label'], test_size=0.3, random_state=42, stratify=df['label']
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# TF-IDF Vectorization (n-grams 1-2 as recommended in roadmap)
print("Vectorizing text with TF-IDF...")
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# Train LinearSVC 
print("Training SVM (LinearSVC)...")
svm_model = LinearSVC(C=1.0, class_weight='balanced', random_state=42)
svm_model.fit(X_train_vec, y_train)

In [ ]:
# Validation performance
y_val_pred = svm_model.predict(X_val_vec)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"\nValidation Accuracy: {val_acc:.4f}")
print("\nClassification Report (Validation):")
print(classification_report(y_val, y_val_pred, target_names=['Real', 'Fake']))

In [ ]:
# Test performance (final baseline)
y_test_pred = svm_model.predict(X_test_vec)
test_acc = accuracy_score(y_test, y_test_pred)
print(f"\nTest Accuracy: {test_acc:.4f}")
print("\nFinal Classification Report (Test):")
print(classification_report(y_test, y_test_pred, target_names=['Real', 'Fake']))
print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, y_test_pred))